<a href="https://colab.research.google.com/github/EsatovinVictoria/ML_Semester5_EsatovinEbenhaezerVictoria/blob/main/Pertemuan_6/Praktikum_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install hnswlib

import pandas as pd
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Load Dataset PBL
df = pd.read_csv('burnout_dataset1.csv')
num_cols = [
    'Age', 'Monthly_Income_USD', 'Work_Hours_Per_Week', 'Job_Satisfaction',
    'Work_Life_Balance', 'Sleep_Hours', 'Anxiety_Score', 'Depression_Score',
    'Mood_Score', 'Emotional_Stability', 'Physical_Activity_Hours',
    'Meditation_Minutes', 'Screen_Time_Hours', 'Social_Media_Hours',
    'Gaming_Hours', 'Coffee_Cups_Per_Day', 'Life_Satisfaction',
    'Productivity_Score', 'Absenteeism_Days', 'Burnout_Score'
]
df_num = df[num_cols].fillna(df[num_cols].mean(numeric_only=True))
scaler = StandardScaler()
X_full = scaler.fit_transform(df_num).astype(np.float32)

configs = [
    {"n_elements": 1000, "dim": 2, "space": "l2", "metric_sklearn": "euclidean"},
    {"n_elements": 1000, "dim": 2, "space": "cosine", "metric_sklearn": "cosine"},
    {"n_elements": 1000, "dim": 5, "space": "l2", "metric_sklearn": "euclidean"},
    {"n_elements": 1000, "dim": 5, "space": "cosine", "metric_sklearn": "cosine"},
    {"n_elements": 50000, "dim": 2, "space": "l2", "metric_sklearn": "euclidean"},
    {"n_elements": 50000, "dim": 2, "space": "cosine", "metric_sklearn": "cosine"},
    {"n_elements": 50000, "dim": 5, "space": "l2", "metric_sklearn": "euclidean"},
    {"n_elements": 50000, "dim": 5, "space": "cosine", "metric_sklearn": "cosine"}
]

k = 5
results = []

print("Memulai eksperimen perbandingan...")

for cfg in configs:
    n_elements = cfg["n_elements"]
    dim = cfg["dim"]
    space = cfg["space"]
    metric_sklearn = cfg["metric_sklearn"]

    print(f"Menjalankan eksperimen: N={n_elements}, Dim={dim}, Metrik={space}...")

    data = X_full[:n_elements, :dim].copy()
    query = data[0:1].copy()

    # 1. Exact NN (Sklearn Brute Force)
    try:
        nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric=metric_sklearn)
        nn.fit(data)
        start = time.time()
        distances_exact, indices_exact = nn.kneighbors(query)
        time_exact = time.time() - start
    except Exception as e:
        time_exact = None
        print(f"Gagal menjalankan Exact NN untuk N={n_elements}: {e}")

    # 2. HNSW
    try:
        p = hnswlib.Index(space=space, dim=dim)
        p.init_index(max_elements=n_elements, ef_construction=100, M=16)
        p.add_items(data)
        p.set_ef(50)

        start = time.time()
        labels_hnsw, distances_hnsw = p.knn_query(query, k=k)
        time_hnsw = time.time() - start
    except Exception as e:
        time_hnsw = None
        print(f"Gagal menjalankan HNSW untuk N={n_elements}: {e}")

    speedup = time_exact / time_hnsw if (time_exact and time_hnsw) else None

    results.append({
        "Jumlah Data": n_elements,
        "Dimensi": dim,
        "Metrik": space,
        "Waktu Exact NN (detik)": f"{time_exact:.6f}" if time_exact is not None else "N/A",
        "Waktu HNSW (detik)": f"{time_hnsw:.6f}" if time_hnsw is not None else "N/A",
        "Speedup (Exact / HNSW)": f"{speedup:.2f}x" if speedup is not None else "N/A"
    })

df_results = pd.DataFrame(results)
import IPython
IPython.display.display(df_results)

Memulai eksperimen perbandingan...
Menjalankan eksperimen: N=1000, Dim=2, Metrik=l2...


Menjalankan eksperimen: N=1000, Dim=2, Metrik=cosine...
Menjalankan eksperimen: N=1000, Dim=5, Metrik=l2...
Menjalankan eksperimen: N=1000, Dim=5, Metrik=cosine...
Menjalankan eksperimen: N=50000, Dim=2, Metrik=l2...


Menjalankan eksperimen: N=50000, Dim=2, Metrik=cosine...


Menjalankan eksperimen: N=50000, Dim=5, Metrik=l2...


Menjalankan eksperimen: N=50000, Dim=5, Metrik=cosine...


,Jumlah Data,Dimensi,Metrik,Waktu Exact NN (detik),Waktu HNSW (detik),Speedup (Exact / HNSW)
0,1000,2,l2,2.302166,0.000111,20676.63x
1,1000,2,cosine,0.003778,0.000060,62.63x
2,1000,5,l2,0.005388,0.000080,67.46x
3,1000,5,cosine,0.002701,0.000070,38.66x
4,50000,2,l2,0.001923,0.000097,19.91x
5,50000,2,cosine,0.003812,0.000067,56.50x
6,50000,5,l2,0.001766,0.000098,18.03x
7,50000,5,cosine,0.003900,0.000072,53.81x
